# 02 SVC Full Training

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`run_svc_full.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_full.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)
- [`run_svc_remaining.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_remaining.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)
- [`run_svc_sw15_followup.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_sw15_followup.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'07_fine_tuning/run_svc_full.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4c2423fc906629e10d321ed1d064bc84e0d35272'}, '07_fine_tuning/run_svc_remaining.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '8ffbf9d591b7b0fd1fc46c56411fb7482ff0c8b1'}, '07_fine_tuning/run_svc_sw15_followup.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '221aaffd7871e87600c61b2d9bb8ffeaf1900a97'}}

## run_svc_full

원본: [07_fine_tuning/run_svc_full.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_full.py)  
Git blob: `4c2423fc906629e10d321ed1d064bc84e0d35272`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

In [ ]:
%%archive_source 07_fine_tuning/run_svc_full.py

import os,sys,time,json,csv,hashlib,traceback
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:os.environ[k]='2'
import numpy as np,joblib,pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support
from threadpoolctl import threadpool_limits
ROOT=Path(__file__).resolve().parent;RT=ROOT/'runtime';OUT=ROOT/'02_SVC'/'full_runs';OUT.mkdir(exist_ok=True)
L='0123456789ABCDEF';base=json.loads((ROOT/'EXPERIMENT_PROTOCOL.json').read_text())['baseline']['SVC']
baseline=np.load(ROOT/'00_baseline/SVC_Combined/reproduced_pred.npy')
x=np.load(RT/'svc_X.npy',mmap_mode='r');y=np.load(RT/'svc_y.npy')
v=np.load(RT/'dev_X.npy');vy=np.load(RT/'dev_y.npy');o=np.load(RT/'official_X.npy');oy=np.load(RT/'official_y.npy')
assert x.shape==(82080,1108) and len(vy)==1280 and len(oy)==19200
def run(name,C=1.,gm=1.,hw=1.,tw=1.):
    target=OUT/name;target.mkdir(exist_ok=True)
    if (target/'metrics.json').exists():return json.loads((target/'metrics.json').read_text())
    params={'condition':name,'C':C,'gamma_multiplier':gm,'handmade_weight':hw,'target_weight':tw,'training_rows':len(y),'seed':42,'official_train_only':True,'new_validation_used_in_fit':False}
    (target/'config.json').write_text(json.dumps(params,indent=2))
    weights=np.ones(len(y));weights[76800:]=hw;weights[76800:][np.isin(y[76800:],[11,13,12])]*=tw
    gamma=float(gm/(x.shape[1]*x.var()))
    m=SVC(C=C,gamma=gamma,cache_size=200)
    t=time.perf_counter();print('START',name,flush=True)
    m.fit(x,y,sample_weight=weights)
    fit=time.perf_counter()-t
    print('FIT_DONE',name,fit,flush=True)
    joblib.dump(m,target/'model.joblib',compress=3)
    pv=m.predict(v)
    np.save(target/'pred_dev.npy',pv)
    pp,rr,ff,_=precision_recall_fscore_support(vy,pv,labels=np.arange(16),zero_division=0)
    met={**params,'gamma':gamma,'fit_s':fit,'correct':int((pv==vy).sum()),'accuracy':float(np.mean(pv==vy)),'macro_precision':float(pp.mean()),'macro_f1':float(ff.mean()),'corrected':int(((baseline!=vy)&(pv==vy)).sum()),'regressed':int(((baseline==vy)&(pv!=vy)).sum()),'recall':dict(zip(L,rr.tolist())),'timing_note':'fit time only; simultaneous separate-core experiment may influence time, not final matched benchmark'}
    (target/'dev_metrics.json').write_text(json.dumps(met,indent=2))
    print('DEV_DONE',name,met['correct'],met['macro_f1'],flush=True)
    # Gate on development accuracy and precision, F1 before expensive official evaluation
    if met['correct']>base['correct'] and met['macro_precision']>=base['precision'] and met['macro_f1']>base['f1']:
        op=m.predict(o);np.save(target/'pred_official.npy',op);met['official_correct']=int((op==oy).sum())
        met['official_accuracy']=float(np.mean(op==oy))
        pp0,rr0,ff0,_=precision_recall_fscore_support(vy,baseline,labels=np.arange(16),zero_division=0)
        met['protected_pass']=all(rr[L.index(c)]>=rr0[L.index(c)]-2/80-1e-12 for c in '086')
        met['target_pass']=float(np.mean(rr[[L.index(c) for c in 'BDC']]))>float(np.mean(rr0[[L.index(c) for c in 'BDC']]))
        met['accuracy_guardrails_pass']=met['official_correct']>=base['official_correct']-19 and met['protected_pass'] and met['target_pass']
    else:
        met['official_correct']=None;met['accuracy_guardrails_pass']=False;met['official_not_run_reason']='failed development accuracy/precision/F1 gate'
    met['final_adoption']=False
    with (target/'model.joblib').open('rb') as f:met['checkpoint_sha256']=hashlib.file_digest(f,'sha256').hexdigest()
    (target/'metrics.json').write_text(json.dumps(met,indent=2))
    print('DONE',name,met['correct'],met['official_correct'],met['accuracy_guardrails_pass'],flush=True)
    return met
if __name__=='__main__':
    with threadpool_limits(2):
        for spec in [('C2_G2',2.,2.,1.,1.),('SW4',1.,1.,4.,1.),('TW2',1.,1.,1.,2.)]:
            try:run(*spec)
            except Exception:
                traceback.print_exc();raise


## run_svc_remaining

원본: [07_fine_tuning/run_svc_remaining.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_remaining.py)  
Git blob: `8ffbf9d591b7b0fd1fc46c56411fb7482ff0c8b1`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

In [ ]:
%%archive_source 07_fine_tuning/run_svc_remaining.py

import os,sys,time,json,csv,hashlib,traceback
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:os.environ[k]='2'
import numpy as np,joblib,pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support
from threadpoolctl import threadpool_limits
ROOT=Path(__file__).resolve().parent;RT=ROOT/'runtime';OUT=ROOT/'02_SVC'/'full_runs';OUT.mkdir(exist_ok=True)
L='0123456789ABCDEF';base=json.loads((ROOT/'EXPERIMENT_PROTOCOL.json').read_text())['baseline']['SVC']
baseline=np.load(ROOT/'00_baseline/SVC_Combined/reproduced_pred.npy')
x=np.load(RT/'svc_X.npy',mmap_mode='r');y=np.load(RT/'svc_y.npy')
v=np.load(RT/'dev_X.npy');vy=np.load(RT/'dev_y.npy');o=np.load(RT/'official_X.npy');oy=np.load(RT/'official_y.npy')
assert x.shape==(82080,1108) and len(vy)==1280 and len(oy)==19200
def fast_svc_ovo_predict(model, data, chunk=128):
    sv=np.asarray(model.support_vectors_,dtype=np.float64)
    norms=np.einsum('ij,ij->i',sv,sv)
    starts=np.r_[0,np.cumsum(model.n_support_)]
    ncl=len(model.classes_)
    ret=[];min_abs=np.inf
    for lo in range(0,len(data),chunk):
        z0=np.asarray(data[lo:lo+chunk],dtype=np.float64)
        sq=np.einsum('ij,ij->i',z0,z0)[:,None]+norms[None,:]-2*z0@sv.T
        np.maximum(sq,0,out=sq);sq*=-model._gamma;np.exp(sq,out=sq)
        votes=np.zeros((len(z0),ncl),np.int16);idx=0
        for i in range(ncl):
            si=slice(starts[i],starts[i+1])
            for j in range(i+1,ncl):
                sj=slice(starts[j],starts[j+1])
                dv0=sq[:,si]@model.dual_coef_[j-1,si]+sq[:,sj]@model.dual_coef_[i,sj]+model.intercept_[idx]
                min_abs=min(min_abs,float(np.min(np.abs(dv0))))
                votes[:,i]+=dv0>0;votes[:,j]+=dv0<=0
                idx+=1
        ret.append(model.classes_[votes.argmax(axis=1)])
    return np.concatenate(ret),min_abs


def run(name,C=1.,gm=1.,hw=1.,tw=1.):
    target=OUT/name;target.mkdir(exist_ok=True)
    if (target/'metrics.json').exists():return json.loads((target/'metrics.json').read_text())
    params={'condition':name,'C':C,'gamma_multiplier':gm,'handmade_weight':hw,'target_weight':tw,'training_rows':len(y),'seed':42,'official_train_only':True,'new_validation_used_in_fit':False}
    (target/'config.json').write_text(json.dumps(params,indent=2))
    weights=np.ones(len(y));weights[76800:]=hw;weights[76800:][np.isin(y[76800:],[11,13,12])]*=tw
    gamma=float(gm/(x.shape[1]*x.var()))
    m=SVC(C=C,gamma=gamma,cache_size=200)
    t=time.perf_counter();print('START',name,flush=True)
    m.fit(x,y,sample_weight=weights)
    fit=time.perf_counter()-t
    print('FIT_DONE',name,fit,flush=True)
    joblib.dump(m,target/'model.joblib',compress=3)
    pv=m.predict(v)
    np.save(target/'pred_dev.npy',pv)
    pp,rr,ff,_=precision_recall_fscore_support(vy,pv,labels=np.arange(16),zero_division=0)
    met={**params,'gamma':gamma,'fit_s':fit,'correct':int((pv==vy).sum()),'accuracy':float(np.mean(pv==vy)),'macro_precision':float(pp.mean()),'macro_f1':float(ff.mean()),'corrected':int(((baseline!=vy)&(pv==vy)).sum()),'regressed':int(((baseline==vy)&(pv!=vy)).sum()),'recall':dict(zip(L,rr.tolist())),'timing_note':'fit time only; simultaneous separate-core experiment may influence time, not final matched benchmark'}
    (target/'dev_metrics.json').write_text(json.dumps(met,indent=2))
    print('DEV_DONE',name,met['correct'],met['macro_f1'],flush=True)
    # Gate on development accuracy and precision, F1 before expensive official evaluation
    if met['correct']>base['correct'] and met['macro_precision']>=base['precision'] and met['macro_f1']>base['f1']:
        op,margin=fast_svc_ovo_predict(m,o);np.save(target/'pred_official.npy',op);met['official_correct']=int((op==oy).sum())
        native_check=m.predict(o[np.arange(0,len(o),30)])
        assert np.array_equal(native_check,op[np.arange(0,len(o),30)])
        check_dev,_=fast_svc_ovo_predict(m,v);assert np.array_equal(check_dev,pv)
        met['official_evaluation_method']='RBF OVO direct formula full 19200, native subset 640 and all 1280 dev predictions matched'
        met['min_abs_pairwise_margin']=margin
        met['native_official_full_completed']=False
        met['official_accuracy']=float(np.mean(op==oy))
        pp0,rr0,ff0,_=precision_recall_fscore_support(vy,baseline,labels=np.arange(16),zero_division=0)
        met['protected_pass']=all(rr[L.index(c)]>=rr0[L.index(c)]-2/80-1e-12 for c in '086')
        met['target_pass']=float(np.mean(rr[[L.index(c) for c in 'BDC']]))>float(np.mean(rr0[[L.index(c) for c in 'BDC']]))
        met['accuracy_guardrails_pass']=met['official_correct']>=base['official_correct']-19 and met['protected_pass'] and met['target_pass']
    else:
        met['official_correct']=None;met['accuracy_guardrails_pass']=False;met['official_not_run_reason']='failed development accuracy/precision/F1 gate'
    met['final_adoption']=False
    with (target/'model.joblib').open('rb') as f:met['checkpoint_sha256']=hashlib.file_digest(f,'sha256').hexdigest()
    (target/'metrics.json').write_text(json.dumps(met,indent=2))
    print('DONE',name,met['correct'],met['official_correct'],met['accuracy_guardrails_pass'],flush=True)
    return met
if __name__=='__main__':
    with threadpool_limits(2):
        for spec in [('TW2',1.,1.,1.,2.)]:
            try:run(*spec)
            except Exception:
                traceback.print_exc();raise


## run_svc_sw15_followup

원본: [07_fine_tuning/run_svc_sw15_followup.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/run_svc_sw15_followup.py)  
Git blob: `221aaffd7871e87600c61b2d9bb8ffeaf1900a97`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

In [ ]:
%%archive_source 07_fine_tuning/run_svc_sw15_followup.py

import os,sys,time,json,csv,hashlib,traceback
from pathlib import Path
for k in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:os.environ[k]='2'
import numpy as np,joblib,pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import precision_recall_fscore_support
from threadpoolctl import threadpool_limits
ROOT=Path(__file__).resolve().parent;RT=ROOT/'runtime';OUT=ROOT/'02_SVC'/'full_runs';OUT.mkdir(exist_ok=True)
L='0123456789ABCDEF';base=json.loads((ROOT/'EXPERIMENT_PROTOCOL.json').read_text())['baseline']['SVC']
baseline=np.load(ROOT/'00_baseline/SVC_Combined/reproduced_pred.npy')
x=np.load(RT/'svc_X.npy',mmap_mode='r');y=np.load(RT/'svc_y.npy')
v=np.load(RT/'dev_X.npy');vy=np.load(RT/'dev_y.npy');o=np.load(RT/'official_X.npy');oy=np.load(RT/'official_y.npy')
assert x.shape==(82080,1108) and len(vy)==1280 and len(oy)==19200
def fast_svc_ovo_predict(model, data, chunk=128):
    sv=np.asarray(model.support_vectors_,dtype=np.float64)
    norms=np.einsum('ij,ij->i',sv,sv)
    starts=np.r_[0,np.cumsum(model.n_support_)]
    ncl=len(model.classes_)
    ret=[];min_abs=np.inf
    for lo in range(0,len(data),chunk):
        z0=np.asarray(data[lo:lo+chunk],dtype=np.float64)
        sq=np.einsum('ij,ij->i',z0,z0)[:,None]+norms[None,:]-2*z0@sv.T
        np.maximum(sq,0,out=sq);sq*=-model._gamma;np.exp(sq,out=sq)
        votes=np.zeros((len(z0),ncl),np.int16);idx=0
        for i in range(ncl):
            si=slice(starts[i],starts[i+1])
            for j in range(i+1,ncl):
                sj=slice(starts[j],starts[j+1])
                dv0=sq[:,si]@model.dual_coef_[j-1,si]+sq[:,sj]@model.dual_coef_[i,sj]+model.intercept_[idx]
                min_abs=min(min_abs,float(np.min(np.abs(dv0))))
                votes[:,i]+=dv0>0;votes[:,j]+=dv0<=0
                idx+=1
        ret.append(model.classes_[votes.argmax(axis=1)])
    return np.concatenate(ret),min_abs


def run(name,C=1.,gm=1.,hw=1.,tw=1.):
    target=OUT/name;target.mkdir(exist_ok=True)
    if (target/'metrics.json').exists():return json.loads((target/'metrics.json').read_text())
    params={'condition':name,'C':C,'gamma_multiplier':gm,'handmade_weight':hw,'target_weight':tw,'training_rows':len(y),'seed':42,'official_train_only':True,'new_validation_used_in_fit':False}
    (target/'config.json').write_text(json.dumps(params,indent=2))
    weights=np.ones(len(y));weights[76800:]=hw;weights[76800:][np.isin(y[76800:],[11,13,12])]*=tw
    gamma=float(gm/(x.shape[1]*x.var()))
    m=SVC(C=C,gamma=gamma,cache_size=200)
    t=time.perf_counter();print('START',name,flush=True)
    m.fit(x,y,sample_weight=weights)
    fit=time.perf_counter()-t
    print('FIT_DONE',name,fit,flush=True)
    joblib.dump(m,target/'model.joblib',compress=3)
    pv=m.predict(v)
    np.save(target/'pred_dev.npy',pv)
    pp,rr,ff,_=precision_recall_fscore_support(vy,pv,labels=np.arange(16),zero_division=0)
    met={**params,'gamma':gamma,'fit_s':fit,'correct':int((pv==vy).sum()),'accuracy':float(np.mean(pv==vy)),'macro_precision':float(pp.mean()),'macro_f1':float(ff.mean()),'corrected':int(((baseline!=vy)&(pv==vy)).sum()),'regressed':int(((baseline==vy)&(pv!=vy)).sum()),'recall':dict(zip(L,rr.tolist())),'timing_note':'fit time only; simultaneous separate-core experiment may influence time, not final matched benchmark'}
    (target/'dev_metrics.json').write_text(json.dumps(met,indent=2))
    print('DEV_DONE',name,met['correct'],met['macro_f1'],flush=True)
    # Gate on development accuracy and precision, F1 before expensive official evaluation
    if met['correct']>base['correct'] and met['macro_precision']>=base['precision'] and met['macro_f1']>base['f1']:
        op,margin=fast_svc_ovo_predict(m,o);np.save(target/'pred_official.npy',op);met['official_correct']=int((op==oy).sum())
        native_check=m.predict(o[np.arange(0,len(o),30)])
        assert np.array_equal(native_check,op[np.arange(0,len(o),30)])
        check_dev,_=fast_svc_ovo_predict(m,v);assert np.array_equal(check_dev,pv)
        met['official_evaluation_method']='RBF OVO direct formula full 19200, native subset 640 and all 1280 dev predictions matched'
        met['min_abs_pairwise_margin']=margin
        met['native_official_full_completed']=False
        met['official_accuracy']=float(np.mean(op==oy))
        pp0,rr0,ff0,_=precision_recall_fscore_support(vy,baseline,labels=np.arange(16),zero_division=0)
        met['protected_pass']=all(rr[L.index(c)]>=rr0[L.index(c)]-2/80-1e-12 for c in '086')
        met['target_pass']=float(np.mean(rr[[L.index(c) for c in 'BDC']]))>float(np.mean(rr0[[L.index(c) for c in 'BDC']]))
        met['accuracy_guardrails_pass']=met['official_correct']>=base['official_correct']-19 and met['protected_pass'] and met['target_pass']
    else:
        met['official_correct']=None;met['accuracy_guardrails_pass']=False;met['official_not_run_reason']='failed development accuracy/precision/F1 gate'
    met['final_adoption']=False
    with (target/'model.joblib').open('rb') as f:met['checkpoint_sha256']=hashlib.file_digest(f,'sha256').hexdigest()
    (target/'metrics.json').write_text(json.dumps(met,indent=2))
    print('DONE',name,met['correct'],met['official_correct'],met['accuracy_guardrails_pass'],flush=True)
    return met
if __name__=='__main__':
    with threadpool_limits(2):
        for spec in [('SW1.5',1.,1.,1.5,1.)]:
            try:run(*spec)
            except Exception:
                traceback.print_exc();raise


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")